In [1]:

# ==========================================================
# CELL 2: IMPORT REQUIRED LIBRARIES
# ==========================================================
#
# Libraries used for:
# - Audio capture
# - Speech detection
# - Speech recognition
# - Gemini interaction
# - Text-to-speech
#
# ==========================================================
import os
import re
import time
import queue
import subprocess
import threading

import torch
import sounddevice as sd
import numpy as np

from collections import deque
from scipy.io.wavfile import write

from faster_whisper import WhisperModel

from silero_vad import (
    load_silero_vad,
    VADIterator
)

from dotenv import load_dotenv
from groq import Groq

C:\Users\Prathamesh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================================
# CELL 3: SYSTEM CONFIGURATION AND MODEL LOADING
# ==========================================================
#
# Initializes:
# - Groq LLM (Llama 3.3 70B)
# - Faster-Whisper Small
# - Silero VAD
#
# Audio Settings:
# - Sample Rate = 16000 Hz
# - Block Size = 512
#
# ==========================================================

# Load environment variables from .env file
load_dotenv()

# ==========================================================
# GROQ CONFIGURATION
# ==========================================================

from groq import Groq

GROQ_API_KEY = os.getenv(
    "GROQ_API_KEY"
)

client = Groq(
    api_key=GROQ_API_KEY
)

MODEL_NAME = "llama-3.3-70b-versatile"

print(f"Groq Model Loaded: {MODEL_NAME}")

# ==========================================================
# AUDIO CONFIGURATION
# ==========================================================

SAMPLE_RATE = 16000
BLOCK_SIZE = 512

PRE_SPEECH_SECONDS = 0.5

# ==========================================================
# LOAD FASTER WHISPER
# ==========================================================

print("Loading Faster Whisper...")

asr_model = WhisperModel(
    "medium",
    device="cuda",
    compute_type="float16"
)

print("Faster Whisper Loaded")

# ==========================================================
# LOAD SILERO VAD
# ==========================================================

print("Loading Silero VAD...")

vad_model = load_silero_vad()

vad_iterator = VADIterator(
    vad_model,
    sampling_rate=SAMPLE_RATE,
    min_silence_duration_ms=800
)

print("Silero VAD Loaded")

Groq Model Loaded: llama-3.3-70b-versatile
Loading Faster Whisper...
Faster Whisper Loaded
Loading Silero VAD...
Silero VAD Loaded


In [3]:
# ==========================================================
# CELL 4: CONTINUOUS MICROPHONE CAPTURE
# ==========================================================
#
# Creates:
# - Audio Queue
# - Audio Callback
# - Input Stream
#
# Audio frames are continuously pushed into a queue
# for VAD processing.
#
# ==========================================================
audio_queue = queue.Queue()

def audio_callback(
    indata,
    frames,
    time_info,
    status
):

    if status:
        if getattr(status, "input_overflow", False):
            print("Warning: audio input overflow")
        else:
            print(status)

    audio_queue.put(
        indata.copy()
    )

pre_buffer = deque(
    maxlen=int(
        (SAMPLE_RATE * PRE_SPEECH_SECONDS)
        / BLOCK_SIZE
    )
)

print("Microphone Ready")


Microphone Ready


In [4]:
# ==========================================================
# CELL 5: SPEECH SEGMENTATION AND TRANSCRIPTION
# ==========================================================
#
# Functions:
#
# 1. collect_speech_chunk()
#    - Detect speech start
#    - Collect speech audio
#    - Detect speech end
#
# 2. transcribe_chunk()
#    - Convert speech to text
#    - Measure ASR latency
#
# ==========================================================
def collect_speech_chunk():

    global pre_buffer

    collected_audio = []

    speech_active = False

    # Clear any stale queued audio before starting a new capture.
    try:
        while True:
            audio_queue.get_nowait()
    except queue.Empty:
        pass

    print("\nListening...")

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=BLOCK_SIZE,
        callback=audio_callback
    ):

        while True:

            chunk = audio_queue.get()

            audio = chunk.flatten()

            pre_buffer.append(audio)

            result = vad_iterator(
                torch.tensor(
                    audio,
                    dtype=torch.float32
                ),
                return_seconds=True
            )

            if result is not None:

                if (
                    "start" in result
                    and not speech_active
                ):

                    speech_active = True

                    collected_audio = list(
                        pre_buffer
                    )

                    print("Speech Started")

                elif (
                    "end" in result
                    and speech_active
                ):

                    print("Speech Ended")

                    speech_active = False

                    audio_chunk = np.concatenate(
                        collected_audio
                    )

                    vad_iterator.reset_states()

                    pre_buffer.clear()

                    return audio_chunk

            if speech_active:

                collected_audio.append(
                    audio
                )


def transcribe_chunk(audio_chunk):

    write(
        "temp_chunk.wav",
        SAMPLE_RATE,
        (
            audio_chunk * 32767
        ).astype(np.int16)
    )

    start_time = time.time()

    segments, _ = asr_model.transcribe(
        "temp_chunk.wav",
        language="en",
        beam_size=5
    )

    transcript = " ".join(
        segment.text
        for segment in segments
    ).strip()

    latency = (
        time.time() - start_time
    )

    return transcript, latency

In [5]:
# ==========================================================
# CELL 6: STREAMING GROQ LLM
# ==========================================================

from collections import deque
import re
import time

conversation_history = deque(maxlen=20)

TECHNICAL_PROMPT = """
You are a technical voice assistant.

Keep responses concise.
Use short spoken sentences.
Never use markdown.
Never use bullet points.
Spell out numbers and dates.
Answer directly.
Keep responses under three sentences whenever possible.
"""

FRIENDLY_PROMPT = """
You are a friendly receptionist voice assistant.

Speak naturally.
Be warm and polite.
Keep responses concise.
Use short spoken sentences.
Never use markdown.
Never use bullet points.
Spell out numbers and dates.
"""

ACTIVE_PROMPT = FRIENDLY_PROMPT


def stream_groq_response(transcript):

    global conversation_history

    messages = [
        {
            "role": "system",
            "content": ACTIVE_PROMPT
        }
    ]

    messages.extend(conversation_history)

    messages.append(
        {
            "role": "user",
            "content": transcript
        }
    )

    llm_start_time = time.time()

    stream = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=0.7,
        max_tokens=150,
        stream=True
    )

    full_response = ""
    sentence_buffer = ""

    ttft = None
    first_token_received = False

    for chunk in stream:

        token = ""

        if (
            chunk.choices
            and chunk.choices[0].delta.content
        ):
            token = chunk.choices[0].delta.content

        if not token:
            continue

        if not first_token_received:

            ttft = (
                time.time()
                - llm_start_time
            )

            first_token_received = True

            print(
                f"\nTTFT: {ttft:.2f} sec"
            )

        full_response += token
        sentence_buffer += token

        if re.search(
            r'[.!?]\s*$',
            sentence_buffer
        ):

            sentence = (
                sentence_buffer
                .strip()
            )

            yield (
                "sentence",
                sentence
            )

            sentence_buffer = ""

    if sentence_buffer.strip():

        yield (
            "sentence",
            sentence_buffer.strip()
        )

    conversation_history.append(
        {
            "role": "user",
            "content": transcript
        }
    )

    conversation_history.append(
        {
            "role": "assistant",
            "content": full_response
        }
    )

    yield (
        "complete",
        full_response
    )

In [6]:
# ==========================================================
# CELL 8: CONVERSATION LOGGING
# ==========================================================

import os
from datetime import datetime

LOG_FILE = os.path.abspath("conversation_log.txt")

print(f"Conversation log file: {LOG_FILE}")

def save_response(
    user_text,
    assistant_text,
    ttft,
    ttfa,
    total_latency
):
    try:

        with open(
            LOG_FILE,
            "a",
            encoding="utf-8"
        ) as f:

            f.write("\n" + "=" * 70 + "\n")

            f.write(
                f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
            )

            f.write(f"USER: {user_text}\n")

            f.write(f"ASSISTANT: {assistant_text}\n")

            f.write(f"TTFT: {ttft:.2f} sec\n")

            f.write(f"TTFA: {ttfa:.2f} sec\n")

            f.write(f"TOTAL_LATENCY: {total_latency:.2f} sec\n")

            f.write("=" * 70 + "\n")

            f.flush()

            os.fsync(f.fileno())

        print(f"\nConversation successfully saved to:\n{LOG_FILE}")

    except Exception as e:

        print(f"\nLogging Error: {e}")

Conversation log file: c:\Users\Prathamesh\OneDrive\Desktop\Minimal_Voice_Loop\conversation_log.txt


In [7]:
# ==========================================================
# CELL 9: STREAMING PIPER TTS
# ==========================================================

import subprocess
import sounddevice as sd
import time

first_audio_time = None
ttfa = None
user_finished_time = None
first_response_audio_played = False


def stream_tts(text):

    global first_audio_time
    global ttfa
    global user_finished_time
    global first_response_audio_played

    try:

        print(
            f"\n[TTS] Speaking: {text}"
        )

        tts_start = time.time()

        process = subprocess.Popen(
            [
                "piper/piper.exe",
                "--model",
                "piper/en_US-amy-medium.onnx",
                "--config",
                "piper/en_US-amy-medium.onnx.json",
                "--output-raw"
            ],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL
        )

        process.stdin.write(
            text.encode("utf-8")
        )

        process.stdin.close()

        SAMPLE_RATE_TTS = 22050
        CHUNK_BYTES = 4096

        first_audio_played = False

        stream = sd.RawOutputStream(
            samplerate=SAMPLE_RATE_TTS,
            channels=1,
            dtype="int16"
        )

        stream.start()

        while True:

            chunk = process.stdout.read(
                CHUNK_BYTES
            )

            if not chunk:
                break

            if not first_audio_played:

                first_audio_played = True

                current_audio_time = time.time()

                if (
                    not first_response_audio_played
                    and user_finished_time is not None
                ):

                    ttfa = (
                        current_audio_time
                        - user_finished_time
                    )

                    first_response_audio_played = True

            stream.write(chunk)

        stream.stop()
        stream.close()

        process.stdout.close()
        process.wait()

        return True

    except Exception as e:

        print(
            f"TTS Error: {e}"
        )

        return False

In [8]:
# ==========================================================
# CELL 10: MAIN LOOP
# ==========================================================

print("\nVoice Assistant Started")

while True:

    print("\n=== New Interaction ===")
    print("Listening for next speech...")

    first_response_audio_played = False
    ttfa = None

    audio_chunk = collect_speech_chunk()

    duration = (
        len(audio_chunk)
        / SAMPLE_RATE
    )

    if duration < 0.5:
        continue

    user_finished_time = time.time()

    transcript, asr_latency = (
        transcribe_chunk(
            audio_chunk
        )
    )

    if not transcript.strip():
        continue

    print("\nUSER:")
    print(transcript)

    print(
        f"\nASR Latency: "
        f"{asr_latency:.2f} sec"
    )

    normalized_transcript = re.sub(
        r'[^a-z0-9 ]+',
        ' ',
        transcript.lower()
    ).strip()

    if normalized_transcript in [
        "stop",
        "exit",
        "quit",
        "goodbye"
    ]:

        print(
            "\nAssistant stopped."
        )

        break

    full_response = ""

    print("\nASSISTANT:")

    ttfa_printed = False

    for event_type, content in (
        stream_groq_response(
            transcript
        )
    ):

        if event_type == "sentence":

            print(content)

            full_response += (
                content + " "
            )

            stream_tts(content)

            if (
                ttfa is not None
                and not ttfa_printed
            ):

                print(
                    f"\nTTFA: "
                    f"{ttfa:.2f} sec"
                )

                ttfa_printed = True

        elif event_type == "complete":

            full_response = content

    total_latency = (
        time.time()
        - user_finished_time
    )

    print(
        f"\nTotal Response Latency: "
        f"{total_latency:.2f} sec"
    )

    save_response(
        transcript,
        full_response,
        ttft if 'ttft' in globals() else 0.0,
        ttfa if ttfa else 0.0,
        total_latency
    )

    print(
        "\nResponses saved to:"
    )

    print("response.txt")
    print("conversation_log.txt")


Voice Assistant Started

=== New Interaction ===
Listening for next speech...

Listening...
Speech Started
Speech Ended

USER:
Hello, how are you today?

ASR Latency: 1.22 sec

ASSISTANT:

TTFT: 0.60 sec
Hi there, I'm doing great, thank you for asking.

[TTS] Speaking: Hi there, I'm doing great, thank you for asking.

TTFA: 2.85 sec
I'm here to help with any questions you may have.

[TTS] Speaking: I'm here to help with any questions you may have.
How can I assist you today?

[TTS] Speaking: How can I assist you today?

Total Response Latency: 14.13 sec

Conversation successfully saved to:
c:\Users\Prathamesh\OneDrive\Desktop\Minimal_Voice_Loop\conversation_log.txt

Responses saved to:
response.txt
conversation_log.txt

=== New Interaction ===
Listening for next speech...

Listening...
Speech Started
Speech Ended

USER:
What is RAG pipeline?

ASR Latency: 2.10 sec

ASSISTANT:

TTFT: 0.28 sec
The RAG pipeline is a risk assessment and governance framework used in Agile methodologies, ty